# GT(gt_schoolnames.csv)와 대조 — school_candidate가 실제 학교명인지 검증

`3.0-extract-candidates.ipynb` -> `3.1-normalize-candidates.ipynb` 순서로 먼저 실행해서 만든
`data/interim/preprocessed_school_candidate.csv`(`preprocessed_candidate` 컬럼 포함)를 입력으로 사용.

In [52]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src" / "utils") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src" / "utils"))

REPO_ROOT

WindowsPath('d:/Study/dongguk_university/dreampath')

In [53]:
import pandas as pd

df = pd.read_csv(
    REPO_ROOT / "data" / "interim" / "preprocessed_school_candidate.csv", encoding="utf-8-sig"
)
# 후보 0건인 행은 NaN으로 읽히므로 방어
df["school_candidate"] = df["school_candidate"].fillna("")
df["preprocessed_candidate"] = df["preprocessed_candidate"].fillna("")
df.shape

(1000, 7)

In [54]:
df.columns

Index(['comment_id', 'comment', 'comment_clean_1', 'comment_noun',
       'school_candidate', 'school_candidate_count', 'preprocessed_candidate'],
      dtype='str')

## preprocessed_candidate가 GT 정식명 목록에 실제로 있는지만 확인

In [55]:
from utils import gt_match

gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
gt_names = set(gt_df["학교명"])

# preprocessed_candidate는 이미 3.1-normalize-candidates.ipynb에서 약어->정식명, 접미사 확장까지
# 끝낸 상태라, 여기선 GT 정식명 목록에 실제로 있는지만 확인하면 됨.
df["gt_match"] = df["preprocessed_candidate"].apply(lambda s: gt_match(s, gt_names))
df["gt_match_count"] = df["gt_match"].apply(lambda s: len(s.split()) if s else 0)
df[["comment", "school_candidate", "preprocessed_candidate", "gt_match", "gt_match_count"]].head(20)

,comment,school_candidate,preprocessed_candidate,gt_match,gt_match_count
0,동국대학교,동국대학교,동국대학교,동국대학교,1
1,이번엔 중동고 차례입니다 🍗,중동고,중동고등학교,중동고등학교,1
2,우리 반/동아리 대표로 서초초 신청합니다!,서초초,서초초등학교,,0
3,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 건국대,잠실중학교 건국대학교,잠실중학교 건국대학교,2
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여대부속초 서초초 보고,이화여자대학교사범대학부속초등학교 서초초등학교 보고등학교,이화여자대학교사범대학부속초등학교,1
5,학식 말고 치킨 먹고 싶어요 인하대학교,인하대학교,인하대학교,인하대학교,1
6,서울공업고.. 오늘만 기다렸어요,서울공업고,서울공업고등학교,서울공업고등학교,1
7,서강대 학생들 모여라,서강대,서강대학교,서강대학교,1
8,동아리방에서 기다릴게요 서초중,서초중,서초중학교,서초중학교,1
9,대치중 학생입니다 대치중 뽑아주세요,대치중,대치중학교,대치중학교,1


GT에 없는 학교들 존재 (ex - 서초초, 대치초) 
- case 1 서울대치초등학교와 대치초등학교의 매칭 불가
- case 2 인하대학교사범대학부속중학교 안에 인하대부속초등학교 처럼 약어 사용 

In [56]:
# gt_match 중 실제 매칭된 것만 최종 정답(ans) 컬럼으로 따로 저장 + 옆에 개수(ans_count)
df["ans"] = df["gt_match"]
df["ans_count"] = df["gt_match_count"]
df[["comment", "ans", "ans_count","preprocessed_candidate"]].head(20)

,comment,ans,ans_count,preprocessed_candidate
0,동국대학교,동국대학교,1,동국대학교
1,이번엔 중동고 차례입니다 🍗,중동고등학교,1,중동고등학교
2,우리 반/동아리 대표로 서초초 신청합니다!,,0,서초초등학교
3,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중학교 건국대학교,2,잠실중학교 건국대학교
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여자대학교사범대학부속초등학교,1,이화여자대학교사범대학부속초등학교 서초초등학교 보고등학교
5,학식 말고 치킨 먹고 싶어요 인하대학교,인하대학교,1,인하대학교
6,서울공업고.. 오늘만 기다렸어요,서울공업고등학교,1,서울공업고등학교
7,서강대 학생들 모여라,서강대학교,1,서강대학교
8,동아리방에서 기다릴게요 서초중,서초중학교,1,서초중학교
9,대치중 학생입니다 대치중 뽑아주세요,대치중학교,1,대치중학교


In [57]:
# school_candidate_count 대비 실제 GT에 매칭된 개수 분포
# (예: count=1인데 gt_match_count=0 -> "연대"처럼 약칭이라 GT엔 없는 케이스)
df["gt_match_count"].value_counts().sort_index()

gt_match_count
0    162
1    789
2     49
Name: count, dtype: int64

## GT 매칭 실패 케이스 — 약어 후보 vs 오검출(일반 명사) 구분 필요

In [58]:
# 0건(gt_match_count==0) 행들의 preprocessed_candidate를 공백 기준으로 펼쳐서 토큰별 빈도 확인
# ("서초초등학교 건국대학교"면 서초초등학교 +1, 건국대학교 +1 각각 카운트)
zero_df = df[df["gt_match_count"] == 0] #미매칭된 애들만 필터링 
zero_tokens = [tok for candidates in zero_df["preprocessed_candidate"] for tok in candidates.split()]

zero_token_counts = pd.Series(zero_tokens).value_counts()
print(f"0건 행 {len(zero_df)}개, 그 안의 고유 미매칭 토큰 {len(zero_token_counts)}개")
zero_token_counts

0건 행 162개, 그 안의 고유 미매칭 토큰 16개


서초초등학교             35
대치초등학교             29
인하대부속초등학교          19
서울공고등학교            16
인하부초등학교            12
서초등학교               8
인하대학교부속초등학교         7
인하대학교부속고등학교         7
최고등학교               7
이화여자대학교부속고등학교       6
인하대학교부속중학교          5
학교                  5
고등학교                4
중학교                 3
인하대학교사범대학부속초등학교     1
이화여자대학교부속초등학교       1
Name: count, dtype: int64

- 학교, 고등학교 이렇게 나오는애들은 어떤 경우인지 확인 필요 
- 서초초등학교를 서초등학교 (아마 정규식에서 "초"라는 글자에서 잘라버렸기 때문에 해당 문제 발생)
- 최고등학교는 잘 걸러짐 

## 발견한 문제 — 부속학교 변형들이 대량 미매칭

위 결과에서 "인하대부속중", "인하부중", "이대부초" 같은 부속학교 축약 변형들이 GT와 매칭이
전혀 안 되는 걸 발견함 (정식명이 "인하대학교사범대학부속중학교"처럼 불규칙해서 접미사 규칙으로
못 뽑힘). `0.1-build-abbreviations.ipynb`의 `MANUAL_ALIASES`에 아래 항목을 추가

In [59]:
# 0.1-build-abbreviations.ipynb의 MANUAL_ALIASES에 추가한 부속학교 예외 
ADDED_AFTER_THIS_RUN = {
    "이화여자대학교사범대학부속초등학교": ["이화여대부속초", "이대부초"],
    "이화여자대학교사범대학부속이화·금란중학교": ["이화여대부속중", "이대부중"],
    "이화여자대학교사범대학부속이화금란고등학교": ["이화여대부속고", "이대부고"],
    "인하대학교사범대학부속중학교": ["인하대부속중", "인하부중", "인하대학교부속중"],
    "인하대학교사범대학부속고등학교": ["인하대부속고", "인하부고", "인하대학교부속고"],
}
ADDED_AFTER_THIS_RUN

{'이화여자대학교사범대학부속초등학교': ['이화여대부속초', '이대부초'],
 '이화여자대학교사범대학부속이화·금란중학교': ['이화여대부속중', '이대부중'],
 '이화여자대학교사범대학부속이화금란고등학교': ['이화여대부속고', '이대부고'],
 '인하대학교사범대학부속중학교': ['인하대부속중', '인하부중', '인하대학교부속중'],
 '인하대학교사범대학부속고등학교': ['인하대부속고', '인하부고', '인하대학교부속고']}

In [60]:
# 최신 gt_schoolnames.csv(부속학교 별칭 반영됨) 기준으로 preprocessed_candidate부터 다시 계산해서
# 여기서 바로 재검증 (원래는 3.1-normalize-candidates.ipynb에서 하는 일이지만, 확인용으로 여기서도 재실행)
from utils import preprocess_candidate, gt_match

gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
gt_names = set(gt_df["학교명"])
alias_to_canonical = {}
if "약어" in gt_df.columns:
    for aliases, name in zip(gt_df["약어"], gt_df["학교명"]):
        if pd.notna(aliases):
            for alias in aliases.split():
                alias_to_canonical[alias] = name


df["preprocessed_candidate"] = df["school_candidate"].apply(
    lambda s: preprocess_candidate(s, alias_to_canonical)
)
df["gt_match"] = df["preprocessed_candidate"].apply(lambda s: gt_match(s, gt_names))
df["gt_match_count"] = df["gt_match"].apply(lambda s: len(s.split()) if s else 0)
df["ans"] = df["gt_match"]
df["ans_count"] = df["gt_match_count"]

print("재매칭 후 분포:")
print(df["gt_match_count"].value_counts().sort_index())
df[["comment", "school_candidate", "preprocessed_candidate", "ans", "ans_count"]].head(20)

재매칭 후 분포:
gt_match_count
0    162
1    789
2     49
Name: count, dtype: int64


,comment,school_candidate,preprocessed_candidate,ans,ans_count
0,동국대학교,동국대학교,동국대학교,동국대학교,1
1,이번엔 중동고 차례입니다 🍗,중동고,중동고등학교,중동고등학교,1
2,우리 반/동아리 대표로 서초초 신청합니다!,서초초,서초초등학교,,0
3,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 건국대,잠실중학교 건국대학교,잠실중학교 건국대학교,2
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여대부속초 서초초 보고,이화여자대학교사범대학부속초등학교 서초초등학교 보고등학교,이화여자대학교사범대학부속초등학교,1
5,학식 말고 치킨 먹고 싶어요 인하대학교,인하대학교,인하대학교,인하대학교,1
6,서울공업고.. 오늘만 기다렸어요,서울공업고,서울공업고등학교,서울공업고등학교,1
7,서강대 학생들 모여라,서강대,서강대학교,서강대학교,1
8,동아리방에서 기다릴게요 서초중,서초중,서초중학교,서초중학교,1
9,대치중 학생입니다 대치중 뽑아주세요,대치중,대치중학교,대치중학교,1


In [61]:
# 0건(gt_match_count==0) 행들의 preprocessed_candidate를 공백 기준으로 펼쳐서 토큰별 빈도 확인
# ("서초초등학교 건국대학교"면 서초초등학교 +1, 건국대학교 +1 각각 카운트)
zero_df = df[df["gt_match_count"] == 0]
zero_tokens = [tok for candidates in zero_df["preprocessed_candidate"] for tok in candidates.split()]

zero_token_counts = pd.Series(zero_tokens).value_counts()
print(f"0건 행 {len(zero_df)}개, 그 안의 고유 미매칭 토큰 {len(zero_token_counts)}개")
zero_token_counts

0건 행 162개, 그 안의 고유 미매칭 토큰 16개


서초초등학교             35
대치초등학교             29
인하대부속초등학교          19
서울공고등학교            16
인하부초등학교            12
서초등학교               8
인하대학교부속초등학교         7
인하대학교부속고등학교         7
최고등학교               7
이화여자대학교부속고등학교       6
인하대학교부속중학교          5
학교                  5
고등학교                4
중학교                 3
인하대학교사범대학부속초등학교     1
이화여자대학교부속초등학교       1
Name: count, dtype: int64

In [62]:
from utils import apply_token_fixes

# 알려진 표기 오류 보정 (실제 학교명인데 접미사 확장이 살짝 어긋난 경우)
TOKEN_FIXES = {
    "서초등학교": "서초초등학교",
    "인하대부속초등학교": "인하대학교부속초등학교",
}

# 접미사만 남아서 특정 학교로 볼 수 없는 순수 오검출 토큰 (이런 것만 버림)
JUNK_TOKENS = {"학교", "고등학교", "중학교", "최고등학교","초등학교"}

# 0건(gt_match_count==0) 행들: 토큰 보정 후 JUNK_TOKENS만 빼고 나머지는 GT 매칭 여부와 상관없이
# 학교명으로 인정해서 ans/ans_count에 채택
fixed_ans = zero_df["preprocessed_candidate"].apply(
    lambda s: apply_token_fixes(s, TOKEN_FIXES, JUNK_TOKENS)
)

df.loc[zero_df.index, "ans"] = fixed_ans
df.loc[zero_df.index, "ans_count"] = df.loc[zero_df.index, "ans"].apply(
    lambda s: len(s.split()) if s else 0
)

df.loc[zero_df.index, ["comment", "preprocessed_candidate", "ans", "ans_count"]]

,comment,preprocessed_candidate,ans,ans_count
2,우리 반/동아리 대표로 서초초 신청합니다!,서초초등학교,서초초등학교,1
10,시험기간엔 서초초 치킨이 답,서초초등학교,서초초등학교,1
13,인하대부속초 / 치킨 / 제발,인하대부속초등학교,인하대학교부속초등학교,1
17,동아리방에서 기다릴게요 대치초,대치초등학교,대치초등학교,1
25,배고픈 인하부초 학생들 살려주세요ㅠㅠ,인하부초등학교,인하부초등학교,1
...,...,...,...,...
963,서초초도 참여 완료!,서초등학교,서초초등학교,1
979,대치초 학생들 야근 말고 치킨 먹자 🙏,대치초등학교,대치초등학교,1
989,대치초 응원단 여기 있습니다!!,대치초등학교,대치초등학교,1
997,배민 이벤트 참여합니다! 인하부초로 보내주세요,인하부초등학교,인하부초등학교,1


In [ ]:
out_path = REPO_ROOT / "data" / "interim" / "gt_match_results.csv"
df[
    [
        "comment_id",
        "comment",
        "comment_noun",
        "school_candidate",
        "preprocessed_candidate",
        "school_candidate_count",
        "gt_match",
        "gt_match_count",
        "ans",
        "ans_count",
       "comment_clean_1",
    ]
].to_csv(out_path, index=False, encoding="utf-8-sig")
out_path

WindowsPath('d:/Study/dongguk_university/dreampath/data/interim/gt_match_results.csv')